# Chapter 6 Exercise Solutions

Each solution uses the default synthetic data and ends with a judgment to document for real data.


In [1]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "ch06_hcp").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "ch06_hcp" / "scripts"))

from run_analysis import load_inputs
from targeting import (
    apply_account_policy, build_account_features, build_call_plan,
    build_decile_summary, build_hcp_actions, build_hcp_features,
    build_territory_summary, compare_naive_and_gated,
)

inputs = load_inputs(ROOT)
hcp_features, _ = build_hcp_features(inputs)
hcp_deciles, _ = build_decile_summary(hcp_features)
account_features = build_account_features(hcp_features, inputs["accounts"])


## Exercise 1: Change the evidence floor


In [2]:
rows = []
policies = {}
for minimum in [5, 10, 15]:
    policy = apply_account_policy(account_features, min_account_patients=minimum)
    policies[minimum] = policy.set_index("account_id")["account_action"]
    counts = policy.account_action.value_counts()
    rows.append({"minimum": minimum, **counts.to_dict()})

print(pd.DataFrame(rows).fillna(0).to_string(index=False))
changed = policies[5].ne(policies[15])
print()
print("Example changed account:", changed[changed].index[0])


 minimum  Maintain  Increase priority  Monitor  Hold contact  Access review
       5        51                 46       45            21              2
      10        51                 43       51            18              2
      15        41                 30       81            12              1

Example changed account: ACC001


**Methods note:** The threshold controls how much sparse account evidence enters the plan. In real data, document the minimum together with count stability, data completeness, and privacy rules.


## Exercise 2: Audit the volume trap


In [3]:
account_targets = apply_account_policy(account_features)
hcp_targets = build_hcp_actions(hcp_deciles, account_targets)
call_plan = build_call_plan(account_targets, hcp_targets)

print(compare_naive_and_gated(hcp_targets, call_plan).to_string(index=False))


                    plan  selected_hcps  contact_permitted  opted_out  opportunity_patients  recent_contacts
Top 30 by patient volume             30                 20         10                   361               32
    Gated near-term plan             30                 30          0                   267               14


**Methods note:** The gated list is executable under current consent and has less recent saturation. Real deployment still needs approved frequency policy, local review, and a defined measurement plan.


## Exercise 3: Rebalance one territory


In [4]:
territory = build_territory_summary(account_targets, call_plan)
total_calls = call_plan.recommended_calls.sum()
t03 = territory.set_index("territory").loc["T03"]
minimum_calls = int((t03.opportunity_share - 0.02) * total_calls + 0.999)
move = minimum_calls - int(t03.recommended_calls)
donor = territory.sort_values("allocation_gap", ascending=False).iloc[0].territory

print(f"Move {move} calls from {donor} to T03.")
print(f"T03 call share after move: {(t03.recommended_calls + move) / total_calls:.1%}")
print(f"T03 opportunity share:     {t03.opportunity_share:.1%}")


Move 4 calls from T04 to T03.
T03 call share after move: 11.0%
T03 opportunity share:     12.1%


**Methods note:** The territory arithmetic identifies the size of the gap. A production reallocation must select permitted HCPs, respect account and representative capacity, and record which lower-priority activity is displaced.
